# Peres–Mermin d=2 — **v4, multi-backend replication**

Your v3 run gave a clean **S = 4.646 ± 0.017 (37.8σ)** on `ibm_marrakesh`. A single run on one device
is a result; the same protocol replicated across **several backends** is a paper-grade result. This
notebook runs the identical verified v3 protocol (sequential measurement, best-qubit-triple pinning,
dynamical decoupling + twirling, 8192 shots) on the **N least-busy backends** and reports a table plus
a combined figure. Nothing to tune. Saves `pm_results_v4.json`.

In [ ]:
# CELL 1 — install
!pip install -q qiskit qiskit-ibm-runtime qiskit-aer

In [ ]:
# CELL 2 — connect
from google.colab import userdata
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
TOKEN=userdata.get("QISKIT_IBM_TOKEN")
try: CRN=userdata.get("QISKIT_IBM_CRN")
except Exception: CRN=None
service=QiskitRuntimeService(token=TOKEN,instance=CRN) if CRN else QiskitRuntimeService(token=TOKEN)
N_BACKENDS=3   # how many devices to replicate across
print("Available:",[b.name for b in service.backends(operational=True,simulator=False)])

In [ ]:
# CELL 3 — protocol + qubit selection (identical to v3; nothing to edit)
from qiskit import QuantumCircuit, transpile, ClassicalRegister, QuantumRegister
import math
OBS={"XI":("X","I"),"IX":("I","X"),"XX":("X","X"),"IY":("I","Y"),"YI":("Y","I"),
     "YY":("Y","Y"),"XY":("X","Y"),"YX":("Y","X"),"ZZ":("Z","Z")}
CTX={"R1":["XI","IX","XX"],"R2":["IY","YI","YY"],"R3":["XY","YX","ZZ"],
     "C1":["XI","IY","XY"],"C2":["IX","YI","YX"],"C3":["XX","YY","ZZ"]}
SIGN={"R1":1,"R2":1,"R3":1,"C1":1,"C2":1,"C3":-1}
def _ctrl(qc,anc,q,P):
    if P=="I": return
    if P=="X": qc.cx(anc,q)
    elif P=="Z": qc.cz(anc,q)
    elif P=="Y": qc.sdg(q); qc.cx(anc,q); qc.s(q)
def _mp(qc,data,anc,cbit,P0,P1):
    qc.h(anc); _ctrl(qc,anc,data[0],P0); _ctrl(qc,anc,data[1],P1); qc.h(anc); qc.measure(anc,cbit); qc.reset(anc)
def context_circuit(name):
    data=QuantumRegister(2,"d"); anc=QuantumRegister(1,"a"); c=ClassicalRegister(3,"c")
    qc=QuantumCircuit(data,anc,c)
    for j,o in enumerate(CTX[name]):
        p0,p1=OBS[o]; _mp(qc,data,anc[0],c[j],p0,p1)
    return qc
def ctx_value(name,counts):
    tot=sum(counts.values()); acc=0.0
    for bits,cnt in counts.items():
        b=bits.replace(" ",""); o=[(-1)**int(b[-1-j]) for j in range(3)]; acc+=o[0]*o[1]*o[2]*cnt
    return acc/tot
def ctx_se(name,counts):
    tot=sum(counts.values()); even=sum(c for b,c in counts.items() if b.replace(' ','').count('1')%2==0)
    p=even/tot; return math.sqrt(max(4*p*(1-p)/tot,0.0))
def best_triple(backend):
    t=backend.target; edges=set(t.build_coupling_map().get_edges()); nq=t.num_qubits
    def ro(q):
        try: return t["measure"][(q,)].error or 0.05
        except Exception: return 0.05
    tw=[g for g in ["cz","ecr","cx"] if g in t.operation_names]
    def ee(a,b):
        for g in tw:
            for pr in [(a,b),(b,a)]:
                try:
                    e=t[g][pr].error
                    if e is not None: return e
                except Exception: pass
        return 0.05
    nbr={q:set() for q in range(nq)}
    for a,b in edges: nbr[a].add(b); nbr[b].add(a)
    best=None; bs=1e9
    for anc in range(nq):
        nb=list(nbr[anc])
        for i in range(len(nb)):
            for j in range(i+1,len(nb)):
                d0,d1=nb[i],nb[j]; s=ro(anc)+ro(d0)+ro(d1)+ee(anc,d0)+ee(anc,d1)
                if s<bs: bs=s; best=[d0,d1,anc]
    return best
print("Protocol ready.")

In [ ]:
# CELL 4 — FREE noisy-sim sanity check
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
_id=AerSimulator(); S=sum(SIGN[n]*ctx_value(n,_id.run(transpile(context_circuit(n),_id),shots=20000).result().get_counts()) for n in CTX)
print(f"ideal S={S:.3f} (must be 6.000)"); assert S>5.9
nm=NoiseModel(); nm.add_all_qubit_quantum_error(depolarizing_error(0.01,1),['h','s','sdg'])
nm.add_all_qubit_quantum_error(depolarizing_error(0.02,2),['cx','cz']); nm.add_all_qubit_readout_error(ReadoutError([[0.98,0.02],[0.02,0.98]]))
_nz=AerSimulator(noise_model=nm); Sn=sum(SIGN[n]*ctx_value(n,_nz.run(transpile(context_circuit(n),_nz),shots=20000).result().get_counts()) for n in CTX)
print(f"noisy S={Sn:.3f} (expect ~4.5; data-dependent)")

In [ ]:
# CELL 5 — replicate across N least-busy backends
import json, datetime, qiskit, qiskit_ibm_runtime
cands=service.backends(operational=True,simulator=False,min_num_qubits=3)
cands=sorted(cands,key=lambda b:b.status().pending_jobs)[:N_BACKENDS]
print("Replicating on:",[b.name for b in cands])
all_res={}
for backend in cands:
    layout=best_triple(backend)
    sampler=Sampler(mode=backend)
    sampler.options.dynamical_decoupling.enable=True
    sampler.options.dynamical_decoupling.sequence_type="XpXm"
    sampler.options.twirling.enable_gates=True; sampler.options.twirling.enable_measure=True
    per={}; S=0.0; Svar=0.0
    for n in CTX:
        isa=transpile(context_circuit(n),backend,initial_layout=layout,optimization_level=1)
        job=sampler.run([isa],shots=8192); counts=job.result()[0].data.c.get_counts()
        v=ctx_value(n,counts); se=ctx_se(n,counts); S+=SIGN[n]*v; Svar+=se*se
        per[n]={"value":round(v,4),"se":round(se,4),"job_id":job.job_id()}
    SE=Svar**0.5
    all_res[backend.name]={"layout":layout,"S":round(S,4),"SE":round(SE,4),"sigma":round((S-4)/SE,2),"per_context":per}
    print(f"  {backend.name}: S={S:.3f} ± {SE:.3f}  ({(S-4)/SE:.1f}σ)  layout={layout}")
# combined inverse-variance weighted mean
import math
num=sum(r["S"]/r["SE"]**2 for r in all_res.values()); den=sum(1/r["SE"]**2 for r in all_res.values())
Scomb=num/den; SEcomb=math.sqrt(1/den)
out={"test":"d2_peres_mermin_multibackend_v4","timestamp":datetime.datetime.now().isoformat(),
     "shots":8192,"backends":all_res,"combined":{"S":round(Scomb,4),"SE":round(SEcomb,4),"sigma":round((Scomb-4)/SEcomb,2)},
     "versions":{"qiskit":qiskit.__version__,"qiskit_ibm_runtime":qiskit_ibm_runtime.__version__}}
json.dump(out,open("pm_results_v4.json","w"),indent=2)
print(f"\nCOMBINED (inverse-variance weighted): S = {Scomb:.4f} ± {SEcomb:.4f}  ({(Scomb-4)/SEcomb:.1f}σ over {len(all_res)} devices)")
print("Saved pm_results_v4.json — download and send it back.")
try:
    from google.colab import files; files.download("pm_results_v4.json")
except Exception: print("(Files sidebar > pm_results_v4.json > Download.)")